In [ ]:
!git clone https://github.com/mmarcotullio/KernelOracle.git
%cd KernelOracle
!git checkout mansheel-updates
!ls

fatal: destination path 'KernelOracle' already exists and is not an empty directory.
/content/KernelOracle
M	tcn/scripts/preprocess.py
M	tcn/utils/metrics.py
Already on 'mansheel-updates'
Your branch is up to date with 'origin/mansheel-updates'.
 data
 data_preprocessing.py
 data_tcn
'Deep task scheduler for the Linux kernel - Project report.pdf'
 LICENSE
 lkp_project.yml
 model.py
 predict0.pdf
 predict10.pdf
 predict11.pdf
 predict12.pdf
 predict13.pdf
 predict14.pdf
 predict1.pdf
 predict2.pdf
 predict3.pdf
 predict4.pdf
 predict5.pdf
 predict6.pdf
 predict7.pdf
 predict8.pdf
 predict9.pdf
 README.md
 tcn
'TCN_Training_&_Evaluation.ipynb'
 train.ipynb
 train.py
 utils.py
 Vagrantfile


In [ ]:
!pip -q install torch pandas numpy scikit-learn matplotlib tqdm

In [ ]:
from tcn.models.tcn import TCNConfig, TCNNextPid
from tcn.utils.data import TraceWindowDataset, Vocab, batch_to_device
from tcn.utils.metrics import top1_accuracy, measure_inference_latency_ms

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

DATA_DIR = "/content/drive/MyDrive/data_tcn"
print("Files in DATA_DIR:", os.listdir(DATA_DIR))

Files in DATA_DIR: ['dataset_meta.json', 'test_seen.csv', 'train.csv', 'test_unseen.csv', 'all.csv', 'cpu_bound_4p.csv', 'cpu_bound_8p.csv', 'io_mixed.csv', 'sysbench_cpu_8t.csv', 'sysbench_memory_4t.csv', 'hackbench_socket_large.csv', 'hackbench_pipe_large.csv']


In [ ]:
!ln -s "/content/drive/MyDrive/data_tcn" data_tcn
!ls -lh data_tcn
!head -n 2 data_tcn/train.csv

lrwxrwxrwx 1 root root 31 Mar  7 07:53 data_tcn -> /content/drive/MyDrive/data_tcn
timestamp,time_diff,task_name,pid,cpu,prev_state,prev_comm,workload_type,trace_id
3710767.341761,0.000000000,code,861190,003,R,swapper/3,cpu_bound_4p,cpu_bound_4p_run1


In [ ]:
import numpy as np
import json
import os


def build_vocab(df, vocab_out_path):
    df["pid_str"] = df["pid"].astype(str)
    unique_pids = sorted(df["pid_str"].unique())
    pid_to_idx = {"<UNK>": 0}
    for i, pid in enumerate(unique_pids, start=1):
        pid_to_idx[pid] = i
    unique_states = sorted(df["prev_state"].astype(str).unique())
    state_to_idx = {state: i for i, state in enumerate(unique_states)}

    vocab = {
        "pid_to_idx": pid_to_idx,
        "state_to_idx": state_to_idx,
    }

    with open(vocab_out_path, "w") as f:
        json.dump(vocab, f)

    return vocab



def load_vocab(vocab_path):
    with open(vocab_path, "r") as f:
        return json.load(f)



def encode_df(df, vocab):


    df["pid_str"] = df["pid"].astype(str)
    pid_to_idx = vocab["pid_to_idx"]
    if "<UNK>" not in pid_to_idx:
        pid_to_idx["<UNK>"] = 0
    unk_idx = pid_to_idx["<UNK>"]
    df["pid_idx"] = (
        df["pid_str"]
        .map(lambda x: pid_to_idx.get(x, unk_idx))
        .astype(np.int64)
    )
    state_to_idx = vocab["state_to_idx"]
    df["state_idx"] = (
        df["prev_state"].astype(str)
        .map(lambda x: state_to_idx.get(x, 0))
        .astype(np.int64)
    )

    return df

In [ ]:
rm -rf tcn/artifacts/*

In [ ]:
%%bash

FILE="tcn/scripts/preprocess.py"


sed -i 's/df\["pid_idx"\] = df\["pid_str"\].map(vocab\["pid_to_idx"\]).astype(np.int64)/pid_to_idx = vocab["pid_to_idx"]\n    if "<UNK>" not in pid_to_idx:\n        pid_to_idx["<UNK>"] = 0\n    unk_idx = pid_to_idx["<UNK>"]\n    df["pid_idx"] = df["pid_str"].map(lambda x: pid_to_idx.get(x, unk_idx)).astype(np.int64)/' $FILE

echo "Patched preprocess.py"

Patched preprocess.py


In [ ]:
!sed -n '35,65p' tcn/scripts/preprocess.py

        "pid_to_idx": pid_to_idx,
        "idx_to_pid": idx_to_pid,
        "state_to_idx": state_to_idx,
        "idx_to_state": idx_to_state,
    }


def encode_df(df: pd.DataFrame, vocab: Dict) -> pd.DataFrame:
    df = df.copy()

    df["pid_str"] = df["pid"].astype(str)
    pid_to_idx = vocab["pid_to_idx"]
    if "<UNK>" not in pid_to_idx:
        pid_to_idx["<UNK>"] = 0
    unk_idx = pid_to_idx["<UNK>"]
    df["pid_idx"] = df["pid_str"].map(lambda x: pid_to_idx.get(x, unk_idx)).astype(np.int64)

    df["prev_state_str"] = df["prev_state"].astype(str).fillna("UNK")

    if "UNK" not in vocab["state_to_idx"]:

        vocab["state_to_idx"]["UNK"] = len(vocab["state_to_idx"])
        vocab["idx_to_state"][vocab["state_to_idx"]["UNK"]] = "UNK"

    df["state_idx"] = df["prev_state_str"].map(lambda s: vocab["state_to_idx"].get(s, vocab["state_to_idx"]["UNK"])).astype(np.int64)


    df["cpu"] = pd.to_numeric(df.get("cpu", 0), errors="coerce").fillna(0.0).astype(np.float32)
    df["tim

In [ ]:
!rm -rf tcn/artifacts/*

In [ ]:
!rm -rf tcn/artifacts/*
!ls -lh tcn/artifacts

total 0


In [ ]:
!rm -rf tcn/artifacts/*

In [ ]:
!python tcn/scripts/preprocess.py \
 --csv data_tcn/train.csv \
 --out tcn/artifacts/train.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4

Saved tcn/artifacts/train.npz
Windows: 1812164, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [ ]:
!python tcn/scripts/preprocess.py \
 --csv data_tcn/test_seen.csv \
 --out tcn/artifacts/test_seen.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4 \
 --use_existing_vocab

Saved tcn/artifacts/test_seen.npz
Windows: 792463, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [ ]:
!python tcn/scripts/preprocess.py \
 --csv data_tcn/test_unseen.csv \
 --out tcn/artifacts/test_unseen.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4 \
 --use_existing_vocab

Saved tcn/artifacts/test_unseen.npz
Windows: 2172560, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [ ]:
for w in workloads:
    csv_path = f"data_tcn/{w}.csv"
    out_path = f"tcn/artifacts/{w}.npz"

    cmd = f"""
    python tcn/scripts/preprocess.py \
        --csv {csv_path} \
        --out {out_path} \
        --vocab_out tcn/artifacts/vocab.json \
        --seq_len 64 \
        --stride 4 \
        --use_existing_vocab
    """

    print(f"\nProcessing workload: {w}")
    os.system(cmd)



Processing workload: cpu_bound_4p

Processing workload: cpu_bound_8p

Processing workload: hackbench_pipe_large

Processing workload: hackbench_socket_large

Processing workload: io_mixed

Processing workload: sysbench_cpu_8t

Processing workload: sysbench_memory_4t


In [ ]:
!ls -lh tcn/artifacts

total 119M
-rw-r--r-- 1 root root 538K Mar  7 09:47 cpu_bound_4p.npz
-rw-r--r-- 1 root root 264K Mar  7 09:47 cpu_bound_8p.npz
-rw-r--r-- 1 root root 9.0M Mar  7 09:48 hackbench_pipe_large.npz
-rw-r--r-- 1 root root 4.3M Mar  7 09:48 hackbench_socket_large.npz
-rw-r--r-- 1 root root 329K Mar  7 09:48 io_mixed.npz
-rw-r--r-- 1 root root 547K Mar  7 09:48 sysbench_cpu_8t.npz
-rw-r--r-- 1 root root 456K Mar  7 09:48 sysbench_memory_4t.npz
-rw-r--r-- 1 root root  16M Mar  7 09:46 test_seen.npz
-rw-r--r-- 1 root root  38M Mar  7 09:47 test_unseen.npz
-rw-r--r-- 1 root root  51M Mar  7 09:46 train.npz
-rw-r--r-- 1 root root 124K Mar  7 09:45 vocab.json


In [ ]:
import sys
sys.path.append("/content/KernelOracle")

In [ ]:
%cd /content/KernelOracle

/content/KernelOracle


In [ ]:
%cd /content/KernelOracle
!git branch

/content/KernelOracle
  main
* mansheel-updates


In [ ]:
%cd /content/KernelOracle

!python -m tcn.train_tcn \
  --train_npz tcn/artifacts/train.npz \
  --test_seen_npz tcn/artifacts/test_seen.npz \
  --test_unseen_npz tcn/artifacts/test_unseen.npz \
  --vocab tcn/artifacts/vocab.json \
  --epochs 5 \
  --batch_size 128 \
  --lr 0.001 \
  --save_dir tcn/checkpoints

/content/KernelOracle
usage: train_tcn.py [-h] [--train_npz TRAIN_NPZ]
                    [--test_seen_npz TEST_SEEN_NPZ]
                    [--test_unseen_npz TEST_UNSEEN_NPZ] [--vocab VOCAB]
                    [--epochs EPOCHS] [--batch_size BATCH_SIZE] [--lr LR]
                    [--channels CHANNELS] [--num_blocks NUM_BLOCKS]
                    [--kernel_size KERNEL_SIZE] [--dropout DROPOUT]
                    [--save_dir SAVE_DIR]
train_tcn.py: error: unrecognized arguments: --test_cpu_bound_4p_npz tcn/artifacts/cpu_bound_4p.npz --test_cpu_bound_8p_npz tcn/artifacts/cpu_bound_8p.npz --test_hackbench_pipe_large_npz tcn/artifacts/hackbench_pipe_large.npz --test_hackbench_socket_large_npz tcn/artifacts/hackbench_socket_large.npz --test_io_mixed_npz tcn/artifacts/io_mixed.npz --test_sysbench_cpu_8t_npz tcn/artifacts/sysbench_cpu_8t.npz --test_sysbench_memory_4t_npz tcn/artifacts/sysbench_memory_4t.npz


In [ ]:
%%bash
sed -i 's/model(batch\["pid"\], batch\["cont"\])/model(batch["pid"], batch["cont"], state=batch["state"])/' tcn/utils/metrics.py
echo "Latency function patched."

Latency function patched.


In [ ]:
import numpy as np

data = np.load("tcn/artifacts/train.npz")
print(data.files)

['pid', 'cont', 'state', 'y', 'seq_len']


In [ ]:
import torch
import time
from tcn.models.tcn import TCNConfig, TCNNextPid
from tcn.utils.data import TraceWindowDataset, Vocab, batch_to_device
from torch.utils.data import DataLoader

device = torch.device("cpu")
torch.set_num_threads(1)  # optional but improves latency stability

print("Using device:", device)
vocab = Vocab.load("tcn/artifacts/vocab.json")

ckpt = torch.load(
    "tcn/checkpoints/tcn_nextpid_best.pt",
    map_location=device
)

cfg = TCNConfig(**ckpt["cfg"])
model = TCNNextPid(cfg).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

ds = TraceWindowDataset("tcn/artifacts/train.npz")
loader = DataLoader(ds, batch_size=128, shuffle=False)

batch = next(iter(loader))
batch = batch_to_device(batch, device)

pid = batch["pid"]
cont = batch["cont"]
state = batch["state"]

print("Model device:", next(model.parameters()).device)
print("pid device:", pid.device)
print("cont device:", cont.device)
print("state device:", state.device)

print("pid shape:", pid.shape)
print("cont shape:", cont.shape)
print("state shape:", state.shape)

Using device: cpu
Model device: cpu
pid device: cpu
cont device: cpu
state device: cpu
pid shape: torch.Size([128, 64])
cont shape: torch.Size([128, 64, 2])
state shape: torch.Size([128, 64])


In [ ]:
warmup = 20
iters = 100

for _ in range(warmup):
    _ = model(pid, cont, state)

start = time.perf_counter()

for _ in range(iters):
    _ = model(pid, cont, state)

end = time.perf_counter()

lat_ms = (end - start) / iters * 1000


batch_size = pid.shape[0]

print(f"\nAvg forward latency per batch: {lat_ms:.3f} ms (CPU)")
print(f"Latency per sample: {(lat_ms / batch_size):.6f} ms")

throughput = batch_size / (lat_ms / 1000)
print(f"Throughput: {throughput:.2f} samples/sec")


Avg forward latency per batch: 170.386 ms (CPU)
Latency per sample: 1.331137 ms
Throughput: 751.24 samples/sec
